In [3]:
import pandas as pd
from sqlalchemy import create_engine, text
import country_converter as coco

#### **1) Creación del motor:**

In [4]:
# 1.1) Configuración de la conexión mediante el motor (engine) usando SQLAlchemy:
USER = "root"
PASSWORD = "yes"
HOST = "localhost"
DATABASE = "motogp_db"

connection_string = f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}/{DATABASE}"
engine = create_engine(connection_string)

# 1.2) Eliminamos los datos de las tablas sin eliminar su estructura para iniciar
# la migración desde cero: 
TRUNCATE = True

# Usamos un bloque with para manejar la conexión:
if TRUNCATE: 
    with engine.begin() as conn:
        # Desactivar temporalmente revisión de llaves foráneas para evitar bloqueos de 
        # seguridad y permitir la limpieza de los datos sin ningún inconveniente:
        conn.execute(text("SET FOREIGN_KEY_CHECKS = 0;"))
        
        # Truncar (vaciar) todas las tablas involucradas:
        conn.execute(text("TRUNCATE TABLE bikes;"))
        conn.execute(text("TRUNCATE TABLE teams;"))
        conn.execute(text("TRUNCATE TABLE categories;"))
        conn.execute(text("TRUNCATE TABLE countries;"))
        conn.execute(text("TRUNCATE TABLE circuits;"))
        conn.execute(text("TRUNCATE TABLE races;"))
        conn.execute(text("TRUNCATE TABLE results;"))
        conn.execute(text("TRUNCATE TABLE riders;"))

        # Volver a activar la revisión de llaves foráneas:
        conn.execute(text("SET FOREIGN_KEY_CHECKS = 1;"))
    
    print("Conexión configurada con éxito. Base de datos vaciada y lista para una nueva migración limpia.")

else: 
    print("Conexión configurada con éxito.")

Conexión configurada con éxito. Base de datos vaciada y lista para una nueva migración limpia.


#### **2) Lectura del CSV:**

In [5]:
# 2.1) Leer el CSV:
df = pd.read_csv("../data/moto_results.csv")

# 2.2) Pre-procesar códigos deportivos (FIM/IOC).
# Algunos códigos deportivos varían del estándar y deben indicarse explícitamente a nombres:
fim_to_standard = {
    'SPA': 'Spain',
    'NED': 'Netherlands',
    'GER': 'Germany',
    'SWI': 'Switzerland',
    'RSA': 'South Africa',
    'POR': 'Portugal',
    'MAL': 'Malaysia',
    'INA': 'Indonesia',
    'RSM': 'San Marino',
    'NZE': 'New Zealand', 
    'CRO': 'Croatia', 
    'SLO': 'Slovenia', 
    'DAN': 'Denmark'
}

df['rider_country'] = df['rider_country'].replace(fim_to_standard)

# 2.3) Usar coco (country_converter) para convertir automáticamente todos los códigos de países a ISO3: 
# 2.3) Extraer solo los países únicos para convertir súper rápido
cc = coco.CountryConverter()

unique_circuits = df['circuit_country'].dropna().unique()
circuit_map = dict(zip(unique_circuits, cc.convert(names=unique_circuits.tolist(), to='ISO3')))
df['circuit_country'] = df['circuit_country'].map(circuit_map)

unique_riders = df['rider_country'].dropna().unique()
rider_map = dict(zip(unique_riders, cc.convert(names=unique_riders.tolist(), to='ISO3')))
df['rider_country'] = df['rider_country'].map(rider_map)

df.head(1)

# A continuación seguiremos las pautas de la integridad referencial para ir creando las tablas
# una a una según las dependencias de cada una de ellas.

,year,sequence,category,rider_first_name,rider_last_name,rider_number,rider_country,team_name,bike,position,points,speed,time,race_name,circuit_name,circuit_country,date
0,2021,16,Moto2,Fermín,Aldeguer,54.0,ESP,+EGO Speed Up,Boscoscuro,16,0,154.4,+37.590,Gran Premio Nolan Del Made In Italy E Dell'Emi...,Misano World Circuit Marco Simoncelli,ITA,2021-10-24


#### **3) Countries:**

In [6]:
# 3.1) Extraer países únicos para la tabla 'countries':
# Usamos 'rider_country' como fuente para id_country:
countries = df[['rider_country']].drop_duplicates().dropna()
countries.columns = ['id_country']

# Añadimos una columna igual que el identificador para el nombre, 
# que se podrá enriquecer mediante los nombres completos postre:
countries['country_name'] = countries.id_country

# 3.2) Insertar en la base de datos:
countries.to_sql('countries', con=engine, if_exists='append', index=False)
print("Países migrados exitosamente.")

Países migrados exitosamente.


#### **4) Categories:**

In [7]:
# 4.1) Extraer categorías únicas para la tabla 'categories':
# Usamos 'category' como fuente para id_country:
categories = df[['category']].drop_duplicates().dropna()
categories.columns = ['id_category']

# Añadimos una columna igual que el identificador para el nombre, 
# que se podrá enriquecer mediante los nombres completos postre:
categories['category_name'] = categories['id_category']

# 4.2) Insertar en la base de datos:
categories.to_sql('categories', con=engine, if_exists='append', index=False)
print("Categorías migradas exitosamente.")

Categorías migradas exitosamente.


#### **5) Teams:**

In [8]:
# 5.1) Extraer equipos únicos para la tabla 'teams':
teams = df[['team_name']].drop_duplicates().dropna()

# 5.2) Insertar en la base de datos:
teams.to_sql('teams', con=engine, if_exists='append', index=False)

# Al no enviar la columna 'id_team', MySQL usará el AUTO_INCREMENT automáticamente.
print("Equipos migrados exitosamente.")

Equipos migrados exitosamente.


#### **6) Bikes:**

In [9]:
# 6.1) Extraer motocicletas únicas para la tabla 'bikes':
bikes = df[['bike']].drop_duplicates().dropna()
bikes.columns = ['bike_name']

# 6.2) Insertar en la base de datos:
bikes.to_sql('bikes', con=engine, if_exists='append', index=False)

# Al no enviar la columna 'id_bike', MySQL usará el AUTO_INCREMENT automáticamente.
print("Motocicletas migradas exitosamente.")

Motocicletas migradas exitosamente.


#### **7) Riders:**

In [10]:
# 7.1) Extraer pilotos únicos para la tabla 'riders':
riders = df[['rider_first_name', 'rider_last_name', 'rider_country']].drop_duplicates().dropna()
riders = riders.rename(columns={'rider_first_name': 'first_name', 
                                'rider_last_name': 'last_name', 
                                'rider_country': 'id_country'})

# 7.2) Insertar en la base de datos:
riders.to_sql('riders', con=engine, if_exists='append', index=False)

# Al no enviar la columna 'id_rider', MySQL usará el AUTO_INCREMENT automáticamente.
print("Pilotos migrados exitosamente.")

Pilotos migrados exitosamente.


#### **8) Circuits:**

In [11]:
# 8.1) Extraer circuitos únicos para la tabla 'circuits':
circuits = df[['circuit_name', 'circuit_country']].drop_duplicates().dropna()
circuits = circuits.rename(columns={'circuit_country': 'id_country'})

# 8.2) Insertar en la base de datos:
circuits.to_sql('circuits', con=engine, if_exists='append', index=False)

# Al no enviar la columna 'id_rider', MySQL usará el AUTO_INCREMENT automáticamente.
print("Circuitos migrados exitosamente.")

Circuitos migrados exitosamente.


In [12]:
# Para las dos últimas tablas de 'races' y 'reesults' necesitamos realizar un mapeo 
# entre los valores de los IDs generados y sus valores en el DataFrame. 

# Para ello, descargamos las tablas actuales para tener los IDs generados por MySQL:
riders_map = pd.read_sql("SELECT id_rider, first_name, last_name FROM riders", con=engine)
teams_map = pd.read_sql("SELECT id_team, team_name FROM teams", con=engine)
bikes_map = pd.read_sql("SELECT id_bike, bike_name FROM bikes", con=engine)
circuits_map = pd.read_sql("SELECT id_circuit, circuit_name FROM circuits", con=engine)

print("Mapas de IDs cargados.")

Mapas de IDs cargados.


#### **9) Races:**

In [13]:
# 9.1) Extraer carreras únicas para la tabla 'races':
races = df[['race_name', 'date', 'year', 'sequence', 'category', 'circuit_name']].drop_duplicates().dropna()

# 9.2) Aplicar el mapeo de circuits_map usando merge: 
races = races.merge(circuits_map, on='circuit_name', how='left')

# 9.3) En el caso de category tenemos directamente su id pues es básicamente su nombre:
races = races.rename(columns={'category': 'id_category'})

# 9.4) Escoger las columnas necesarias e insertar en la base de datos:
races = races[['race_name', 'date', 'year', 'sequence', 'id_category', 'id_circuit']]
races.to_sql('races', con=engine, if_exists='append', index=False)

# 9.5) Creamos el mapa de carreras para la tabla final de results:
races_map = pd.read_sql("SELECT id_race, year, sequence, id_category FROM races", con=engine)

# Al no enviar la columna 'id_race', MySQL usará el AUTO_INCREMENT automáticamente.
print("Carreras migradas exitosamente.")

Carreras migradas exitosamente.


#### **10) Results:**

In [14]:
# 10.1) Extraer resultados únicos para la tabla 'results':
results = df[['speed', 'points', 'rider_number', 'time', 'position', 'rider_first_name', 
              'rider_last_name', 'bike', 'team_name', 'year', 'sequence', 'category']].drop_duplicates().dropna()

# 10.2) Mapear el id_race: 
results = results.merge(races_map, left_on=['year', 'sequence', 'category'], 
                        right_on=['year', 'sequence', 'id_category'], how='left')

# 10.3) Mapear el id_rider: 
results = results.merge(riders_map, left_on=['rider_first_name', 'rider_last_name'], 
                        right_on=['first_name', 'last_name'])

# 10.4) Mapear el id_bike: 
results = results.merge(bikes_map, left_on='bike', right_on='bike_name')

# 10.5) Mapear el id_team: 
results = results.merge(teams_map, on='team_name')

# 10.6) Escoger las columnas necesarias e insertar en la base de datos:
results = results[['speed', 'points', 'rider_number', 'time', 'position', 
                   'id_rider', 'id_bike', 'id_team', 'id_race']]
results.to_sql('results', con=engine, if_exists='append', index=False)

# Al no enviar la columna 'id_result', MySQL usará el AUTO_INCREMENT automáticamente.
print("Resultados migrados exitosamente.")

Resultados migrados exitosamente.


In [15]:
# Comprobación: 
query = "SELECT * FROM results;"
resultado = pd.read_sql(query, con=engine)
resultado

,id_result,id_race,id_rider,id_team,id_bike,rider_number,position,points,speed,time
0,1,1,1,1,1,54,16,0.0,154.4,+37.590
1,2,1,2,2,2,14,-1,0.0,154.5,9 Laps
2,3,1,3,3,2,64,12,4.0,155.2,+24.309
3,4,1,4,4,2,6,-1,0.0,154.2,17 Laps
4,5,1,5,2,2,23,15,1.0,154.5,+36.240
...,...,...,...,...,...,...,...,...,...,...
23944,23945,890,684,60,5,3,7,9.0,155.0,+30.618
23945,23946,890,310,965,11,5,9,7.0,154.6,+37.608
23946,23947,890,282,960,5,33,3,16.0,155.7,+18.460
23947,23948,890,366,964,89,77,16,0.0,148.8,1 Lap
